# 04g — XGBoost, LightGBM et Random Forest

Comparaison simple, sur les **mêmes 31 features** que le 04f et les mêmes plis par moteur. Chaque modèle a ses hyperparamètres réglés par Optuna (60 essais pour XGBoost au 04f, 40 pour les deux autres, réglés sur les mêmes plis de recherche).

Résultat : LightGBM fait jeu égal avec XGBoost, Random Forest est nettement moins bon. Aucun ne justifie de changer le modèle final du 04f.


In [ ]:
import sys
sys.path.insert(0, "..")  # pour importer src/ depuis notebooks/

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    max_error, mean_absolute_error, median_absolute_error, r2_score,
)
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5         # fenetre de lissage (Phase 1)
PLAFOND_RUL = 125   # plafond de la RUL (Phase 2)
K_INIT = 20         # cycles servant a definir l'etat initial d'un moteur (04e)
GRAINE = 42

# --- train : capteurs constants retires, min-max, lissage, RUL plafonnee (memes etapes que la Phase 3) ---
train = load_fd001("train")
capteurs_constants = find_constant_sensors(train, get_sensor_columns(train.columns))
train = train.drop(columns=capteurs_constants)
capteurs = get_sensor_columns(train.columns)  # les 15 capteurs utiles
train, scaler = normalize_minmax(train, capteurs)
train = lisser_capteurs(train, capteurs, fenetre=FENETRE)
train = calculer_rul(train)
train = plafonner_rul(train, plafond=PLAFOND_RUL)


def ecart_etat_initial(df, capteurs, k):
    """
    Pour chaque capteur : valeur lissee du cycle courant moins la moyenne des k premiers cycles du moteur.

    Parameters
    ----------
    df : pandas.DataFrame
        Tableau trie par moteur puis par cycle, capteurs deja lisses.
    capteurs : list de str
        Noms des capteurs.
    k : int
        Nombre de cycles de reference ; avant le cycle k, moyenne des cycles connus.

    Returns
    -------
    pandas.DataFrame
        Une colonne "<capteur>|ecartinit_<k>" par capteur, meme index que df.
    """
    unites = df["unit_number"].values
    colonnes = {}
    for capteur in capteurs:
        colonnes[f"{capteur}|ecartinit_{k}"] = np.zeros(len(df))  # espace reserve
    for moteur_id in np.unique(unites):  # un moteur a la fois : jamais de melange entre moteurs
        positions = np.where(unites == moteur_id)[0]
        for capteur in capteurs:
            valeurs = df[capteur].values[positions]
            somme_cumulee = np.cumsum(valeurs)
            moyenne_debut = somme_cumulee / np.arange(1, len(valeurs) + 1)  # cycle < k : moyenne des cycles connus
            if len(valeurs) >= k:
                moyenne_debut[k:] = somme_cumulee[k - 1] / k                # cycle >= k : moyenne figee des k premiers
            colonnes[f"{capteur}|ecartinit_{k}"][positions] = valeurs - moyenne_debut
    return pd.DataFrame(colonnes, index=df.index)


def features_31(df):
    """Jeu retenu au 04e : 15 capteurs lisses, time_cycles, 15 ecarts a l'etat initial."""
    return pd.concat([df[capteurs + ["time_cycles"]], ecart_etat_initial(df, capteurs, K_INIT)], axis=1)


X31 = features_31(train)
assert X31.shape[1] == 31
y = train["RUL_cappee"]
groups = train["unit_number"]
plis = list(GroupKFold(n_splits=5, shuffle=True, random_state=GRAINE).split(X31, y, groups=groups))  # 5 plis par moteur

print(f"{X31.shape[1]} features, 5 plis par moteur.")


In [ ]:
# --- Comparaison : validation sur 5 decoupages par moteur neufs, puis test lu une seule fois par modele ---
def nasa_moyen(y_true, y_pred):
    """Score NASA moyen par prediction : surestimer (ecart > 0) coute plus cher que sous-estimer."""
    ecart = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.mean(np.where(ecart < 0, np.exp(-ecart / 13) - 1, np.exp(ecart / 10) - 1)))


def metriques(y_true, y_pred):
    """Metriques de regression du notebook 04a (celles gardees)."""
    ecarts = y_pred - y_true  # positif = surestimation
    return {
        "RMSE": float(np.sqrt(np.mean(ecarts ** 2))), "MAE": mean_absolute_error(y_true, y_pred), "R2": r2_score(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred), "MaxErr": max_error(y_true, y_pred), "Biais": float(np.mean(ecarts)),
        "NASA_moy": nasa_moyen(y_true, y_pred),
    }


# hyperparametres issus des recherches Optuna (XGBoost : 04f ; LightGBM et Random Forest : 40 essais, TPE graine 42, plis du 04f)
MODELES = {
    "XGBoost": lambda: XGBRegressor(
        random_state=GRAINE, max_depth=4, learning_rate=0.0633, n_estimators=422, subsample=0.5789,
        colsample_bytree=0.8676, min_child_weight=1.3287, reg_lambda=29.1205),
    "LightGBM": lambda: LGBMRegressor(
        random_state=GRAINE, verbose=-1, n_estimators=549, learning_rate=0.02744, num_leaves=8, min_child_samples=72,
        subsample=0.6534, subsample_freq=1, colsample_bytree=0.9839, reg_lambda=2.5794),
    "Random Forest": lambda: RandomForestRegressor(
        random_state=GRAINE, n_jobs=-1, n_estimators=391, max_depth=18, min_samples_leaf=3, max_features=0.5432),
}

# --- validation : memes plis pour les 3 modeles, 5 graines jamais utilisees pour la recherche ---
rmse_val = {nom: [] for nom in MODELES}
nasa_val = {nom: [] for nom in MODELES}
for graine in range(1000, 1005):
    plis_neufs = list(GroupKFold(n_splits=5, shuffle=True, random_state=graine).split(X31, y, groups=groups))
    for nom, fabrique in MODELES.items():
        erreurs_carre, nasas = [], []
        for idx_train, idx_val in plis_neufs:
            pred = fabrique().fit(X31.iloc[idx_train], y.iloc[idx_train]).predict(X31.iloc[idx_val])
            y_val = y.iloc[idx_val].values
            erreurs_carre.append(np.mean((y_val - pred) ** 2))
            nasas.append(nasa_moyen(y_val, pred))
        rmse_val[nom].append(np.sqrt(np.mean(erreurs_carre)))
        nasa_val[nom].append(np.mean(nasas))

# --- test officiel : dernier cycle de chaque moteur, vraie RUL dans RUL_FD001.txt ---
test = load_fd001("test").drop(columns=capteurs_constants)
test[capteurs] = scaler.transform(test[capteurs])  # transform seul : scaler appris sur le train
test = lisser_capteurs(test, capteurs, fenetre=FENETRE)
derniers = test.groupby("unit_number")["time_cycles"].idxmax().sort_index().values
y_test = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None)[0].values.astype(float)
X_test = features_31(test).loc[derniers]

lignes = []
predictions_test = {}
modeles_finaux = {}  # modeles entraines sur tout le train, reutilises par la figure
for nom, fabrique in MODELES.items():
    modeles_finaux[nom] = fabrique().fit(X31, y)
    pred_test = np.clip(modeles_finaux[nom].predict(X_test), 0, None)  # RUL negative sans sens physique
    predictions_test[nom] = pred_test
    ecart_rmse = np.array(rmse_val[nom]) - np.array(rmse_val["XGBoost"])
    lignes.append({
        "modele": nom, "RMSE val": np.mean(rmse_val[nom]), "NASA moy val": np.mean(nasa_val[nom]),
        "ecart RMSE vs XGBoost": ecart_rmse.mean(), "plus bas que XGBoost": f"{int((ecart_rmse < 0).sum())}/5",
        **{f"{m} test": v for m, v in metriques(y_test, pred_test).items()},
    })
pd.set_option("display.width", 250)
print(pd.DataFrame(lignes).round(3).to_string(index=False))


In [ ]:
# --- Graphique : RUL predite cycle apres cycle par les 3 modeles, sur 4 moteurs du test ---
# Lissage et ecart a l'etat initial sont causaux : la prediction au cycle t n'utilise que les cycles <= t.
import matplotlib.pyplot as plt

SEUIL_ALERTE = 30
COULEURS = {"XGBoost": "#56B4E9", "LightGBM": "#E69F00", "Random Forest": "#0072B2"}  # memes couleurs que l'app

X_test_complet = features_31(test)
fin = test.groupby("unit_number")["time_cycles"].transform("max")
rul_finale = pd.Series(y_test, index=np.arange(1, len(y_test) + 1))  # RUL vraie au dernier cycle, par moteur
test["RUL_vraie"] = test["unit_number"].map(rul_finale) + fin - test["time_cycles"]
pred_cycles = {nom: np.clip(m.predict(X_test_complet), 0, None) for nom, m in modeles_finaux.items()}

# 4 moteurs a l'historique long (>= 150 cycles), RUL finale la plus proche de 10, 30, 60 et 100 cycles
longueurs = test.groupby("unit_number")["time_cycles"].max()
candidats = rul_finale[longueurs >= 150]
moteurs = []
for cible in [10, 30, 60, 100]:
    restants = candidats.drop(moteurs)
    moteurs.append(int((restants - cible).abs().idxmin()))

fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharey=True)
for ax, moteur in zip(axes.flat, moteurs):
    masque = (test["unit_number"] == moteur).values
    cycles = test.loc[masque, "time_cycles"].values
    rul_vraie = test.loc[masque, "RUL_vraie"].values
    ax.axhspan(0, SEUIL_ALERTE, color="#D62728", alpha=0.08, linewidth=0)
    ax.axhline(SEUIL_ALERTE, color="#D62728", linestyle="--", linewidth=1.2)
    ax.text(cycles[0], SEUIL_ALERTE + 3, f"seuil d'alerte : {SEUIL_ALERTE} cycles", color="#D62728", fontsize=9)
    ax.plot(cycles, rul_vraie, color="#111111", linewidth=2.5, label="RUL vraie")
    ax.plot(cycles, np.minimum(rul_vraie, PLAFOND_RUL), color="#7F7F7F", linewidth=2, linestyle=":",
            label=f"RUL vraie plafonnee a {PLAFOND_RUL} (cible apprise)")
    for nom, pred in pred_cycles.items():
        ax.plot(cycles, pred[masque], color=COULEURS[nom], linewidth=1.8, label=nom)
        ax.scatter(cycles[-1], pred[masque][-1], color=COULEURS[nom], s=45, edgecolors="white", zorder=3)
    ax.axvline(cycles[-1], color="#555555", linestyle=":", linewidth=1.2)
    erreurs = ", ".join(f"{nom} {pred[masque][-1] - rul_vraie[-1]:+.0f}" for nom, pred in pred_cycles.items())
    ax.set_title(f"Moteur test {moteur} : RUL vraie au dernier cycle = {rul_vraie[-1]:.0f}\necart final : {erreurs}",
                 fontsize=10)
    ax.set_xlabel("Cycle (vol) recu")
    ax.set_ylabel("RUL (cycles restants)")
    ax.set_ylim(0, test.loc[test["unit_number"].isin(moteurs), "RUL_vraie"].max() + 10)  # meme echelle, RUL vraie entiere
    ax.grid(color="#E5E5E5")
axes.flat[0].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()
